<a href="https://colab.research.google.com/github/murilosenati-hash/semaforo-inteligente-visao/blob/main/SmartTrafic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import asyncio
import time
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import List, Optional

# 1. Definição estrita dos Estados do Semáforo
class SignalState(Enum):
    RED = auto()
    GREEN = auto()
    YELLOW = auto()
    EMERGENCY_OVERRIDE = auto()

# 2. Classes de entidades que a IA da câmera consegue classificar
class EntityClass(Enum):
    VEHICLE = "vehicle"
    PEDESTRIAN = "pedestrian"
    BICYCLE = "bicycle"
    EMERGENCY = "emergency"

# 3. Caixa delimitadora (Bounding Box) do objeto na imagem
@dataclass(frozen=True)
class BoundingBox:
    x_min: float
    y_min: float
    x_max: float
    y_max: float

    @property
    def area(self) -> float:
        return (self.x_max - self.x_min) * (self.y_max - self.y_min)

# 4. Entidade rastreada pela visão computacional
@dataclass
class TrackedEntity:
    track_id: int
    entity_class: EntityClass
    confidence: float
    bbox: BoundingBox
    first_detected: float = field(default_factory=time.time)

# 5. O pacote de dados enviado pela câmera a cada segundo
@dataclass
class TelemetryFrame:
    timestamp: float
    entities: List[TrackedEntity]

    def count_valid(self, target: EntityClass, min_confidence: float = 0.70) -> int:
        """Filtra e conta apenas detecções confiáveis da IA"""
        return sum(
            1 for e in self.entities
            if e.entity_class == target and e.confidence >= min_confidence
        )

# 6. O Cérebro: Controlador de Tráfego Adaptativo
class AdaptiveTrafficController:
    def __init__(self, min_green_sec: float = 5.0, max_green_sec: float = 35.0):
        self.min_green = min_green_sec
        self.max_green = max_green_sec
        self.current_state: SignalState = SignalState.RED
        self.last_state_change: float = time.time()
        self.telemetry_queue: asyncio.Queue[TelemetryFrame] = asyncio.Queue(maxsize=50)

    def calculate_dynamic_green(self, frame: TelemetryFrame) -> float:
        """Calcula o tempo ideal de sinal verde com base no fluxo real"""
        vehicles = frame.count_valid(EntityClass.VEHICLE)
        pedestrians = frame.count_valid(EntityClass.PEDESTRIAN)

        # Heurística: 5s base + 1.5s por carro + 2.0s por pedestre
        demanded_time = self.min_green + (vehicles * 1.5) + (pedestrians * 2.0)

        # Trava matemática entre o mínimo e o máximo permitido por lei
        return min(max(demanded_time, self.min_green), self.max_green)

    async def process_telemetry(self, frame: TelemetryFrame) -> None:
        """Recebe os dados da câmera de forma não-bloqueante"""
        now = time.time()
        elapsed = now - self.last_state_change

        # REGRA CRÍTICA 1: Prioridade Máxima para Ambulâncias / Bombeiros
        if frame.count_valid(EntityClass.EMERGENCY) > 0:
            if self.current_state != SignalState.EMERGENCY_OVERRIDE:
                print(f"🚨 [EMERGÊNCIA] Veículo prioritário detectado! Forçando abertura imediata.")
                self.current_state = SignalState.EMERGENCY_OVERRIDE
                self.last_state_change = now
            return

        # MÁQUINA DE ESTADOS:
        if self.current_state == SignalState.RED:
            # Se tem veículos ou pedestres esperando e já deu tempo de segurança
            has_demand = (frame.count_valid(EntityClass.VEHICLE) > 0 or
                          frame.count_valid(EntityClass.PEDESTRIAN) > 0)

            if has_demand and elapsed >= 3.0:
                self.current_state = SignalState.GREEN
                self.last_state_change = now
                green_target = self.calculate_dynamic_green(frame)
                print(f"🟢 [SINAL VERDE] Demanda detectada. Tempo concedido: {green_target:.1f}s")

        elif self.current_state == SignalState.GREEN:
            green_target = self.calculate_dynamic_green(frame)
            if elapsed >= green_target:
                self.current_state = SignalState.YELLOW
                self.last_state_change = now
                print(f"🟡 [SINAL AMARELO] Tempo esgotado ({elapsed:.1f}s). Transição de segurança.")

        elif self.current_state == SignalState.YELLOW:
            # Amarelo fixo de 3 segundos para desaceleração
            if elapsed >= 3.0:
                self.current_state = SignalState.RED
                self.last_state_change = now
                print(f"🔴 [SINAL VERMELHO] Via fechada. Aguardando novo ciclo.")

        elif self.current_state == SignalState.EMERGENCY_OVERRIDE:
            # Se a emergência já passou da faixa da câmera, volta ao normal
            if frame.count_valid(EntityClass.EMERGENCY) == 0:
                print("🔄 [NORMALIZAÇÃO] Veículo de emergência liberou a via. Retornando ao amarelo.")
                self.current_state = SignalState.YELLOW
                self.last_state_change = now

    async def start_camera_pipeline(self):
        """Simulação contínua do fluxo de dados da câmera"""
        print("🚦 Sistema Operacional do Semáforo iniciado.")
        while True:
            frame = await self.telemetry_queue.get()
            await self.process_telemetry(frame)
            self.telemetry_queue.task_done()
            await asyncio.sleep(0.05)


# --- TESTE PRÁTICO DO SISTEMA ---
async def main():
    controller = AdaptiveTrafficController()

    # Inicia o semáforo em segundo plano
    loop_task = asyncio.create_task(controller.start_camera_pipeline())

    # Cenário simulado: Câmera detecta 4 carros e 2 pedestres na rua
    mock_entities = [
        TrackedEntity(1, EntityClass.VEHICLE, 0.94, BoundingBox(10, 20, 50, 60)),
        TrackedEntity(2, EntityClass.VEHICLE, 0.88, BoundingBox(60, 20, 100, 60)),
        TrackedEntity(3, EntityClass.VEHICLE, 0.91, BoundingBox(110, 20, 150, 60)),
        TrackedEntity(4, EntityClass.VEHICLE, 0.79, BoundingBox(160, 20, 200, 60)),
        TrackedEntity(5, EntityClass.PEDESTRIAN, 0.85, BoundingBox(5, 5, 15, 25)),
        TrackedEntity(6, EntityClass.PEDESTRIAN, 0.92, BoundingBox(20, 5, 30, 25)),
    ]
    frame_rua = TelemetryFrame(time.time(), mock_entities)

    # Envia o frame para a fila do semáforo
    await controller.telemetry_queue.put(frame_rua)
    await asyncio.sleep(1.0)

    loop_task.cancel()

if __name__ == "__main__":
    await main()

🚦 Sistema Operacional do Semáforo iniciado.
